# Regenerate Map Figures

Loads pre-saved CSVs from `results/tables/` and regenerates four choropleth maps with:
- Border linewidth: 0.8
- DPI: 300
- Title size: 17
- Output format: SVG

**Maps:** county_mean_neighbor_jsd, county_mean_surprisal, eda_county_reliability, county_jsd_baseline

**Inputs:** saved tables under `results/tables/`. **Outputs:** four SVG figures. **Next:** these presentation maps support the report and frontend assets.

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial.distance import jensenshannon

# Paths: walk up to find project root
PROJECT_ROOT = Path.cwd()
for _ in range(6):
    if (PROJECT_ROOT / 'results').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS = PROJECT_ROOT / 'results' / 'tables'
FIGURES_EDA = PROJECT_ROOT / 'figures' / 'eda' / 'in_depth_analysis'
FIGURES_METHOD = PROJECT_ROOT / 'figures' / 'method_comparison'
FIGURES_EDA.mkdir(parents=True, exist_ok=True)
FIGURES_METHOD.mkdir(parents=True, exist_ok=True)

# Shared plot settings
BORDER_LW = 0.8
TITLE_SIZE = 20
TITLE_SIZE_DUAL = 17
DPI = 300
EDGE_COLOR = '#000000'

# Load CA counties (Census TIGER)
counties = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip")
ca = counties[counties["STATEFP"] == "06"].copy()
ca["county_fips"] = ca["GEOID"]

## 1. County-Level Mean Neighbor JSD

Uses `bayesian_shrinkage_aggregated_counts.csv` + `ca_county_neighbors.csv` to compute raw neighbor JSD (no color merging).

In [32]:
# Pre-pooling: compute raw neighbor JSD from bayesian counts (no color merging)
LAPLACE = 1
MIN_SUPPORT = 30

bay_agg = RESULTS / 'bayesian_shrinkage' / 'bayesian_shrinkage_aggregated_counts.csv'
neighbors_path = PROJECT_ROOT / 'dataset' / 'ca_county_neighbors.csv'
if not neighbors_path.exists():
    neighbors_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'ca_county_neighbors.csv'

df_counts = pd.read_csv(bay_agg)
df_counts['fips'] = df_counts['fips'].astype(str).str.zfill(5)
neighbors = pd.read_csv(neighbors_path)
neighbors['county_fips'] = neighbors['county_fips'].astype(str).str.zfill(5)
neighbors['neighbor_fips'] = neighbors['neighbor_fips'].astype(str).str.zfill(5)
adjacency = list(zip(neighbors['county_fips'], neighbors['neighbor_fips']))
adjacency = [(a, b) if a < b else (b, a) for a, b in adjacency]
adjacency = list(set(adjacency))

all_colors = sorted(df_counts['clr'].unique())
all_lc = sorted(df_counts['lc_type'].unique())
clr_counts = df_counts.groupby(['fips', 'lc_type', 'clr'])['count'].sum().reset_index()
support = clr_counts.groupby(['fips', 'lc_type'])['count'].sum().reset_index()
support_dict = dict(zip(zip(support['fips'], support['lc_type']), support['count']))

def get_dist_raw(fips, lc):
    sub = clr_counts[(clr_counts['fips'] == fips) & (clr_counts['lc_type'] == lc)]
    cnt = dict(zip(sub['clr'], sub['count']))
    vec = np.array([cnt.get(c, 0) + LAPLACE for c in all_colors], dtype=float)
    return vec / vec.sum()

results_raw = []
for fips_a, fips_b in adjacency:
    pair_jsd_raw, pair_supp = [], []
    for lc in all_lc:
        supp_a = support_dict.get((fips_a, lc), 0)
        supp_b = support_dict.get((fips_b, lc), 0)
        if supp_a < MIN_SUPPORT or supp_b < MIN_SUPPORT:
            continue
        d_a = get_dist_raw(fips_a, lc)
        d_b = get_dist_raw(fips_b, lc)
        s = min(supp_a, supp_b)
        pair_jsd_raw.append(jensenshannon(d_a, d_b))
        pair_supp.append(s)
    if pair_jsd_raw:
        w_raw = sum(j * s for j, s in zip(pair_jsd_raw, pair_supp)) / sum(pair_supp)
        results_raw.append({'fips_a': fips_a, 'fips_b': fips_b, 'weighted_jsd': w_raw})

county_jsd_list = {}
for r in results_raw:
    for f in [r['fips_a'], r['fips_b']]:
        county_jsd_list.setdefault(f, []).append(r['weighted_jsd'])
county_mean_jsd = pd.DataFrame([{'fips': f, 'mean_jsd': np.mean(v)} for f, v in county_jsd_list.items()])
county_mean_jsd['fips'] = county_mean_jsd['fips'].astype(str).str.zfill(5)

gdf = ca.merge(county_mean_jsd, left_on='county_fips', right_on='fips', how='left')
gdf['mean_jsd'] = gdf['mean_jsd'].fillna(gdf['mean_jsd'].median())

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column='mean_jsd', ax=ax, cmap='YlOrRd', legend=True, edgecolor=EDGE_COLOR, linewidth=BORDER_LW,
         legend_kwds={'label': 'Mean neighbor JSD'})
ax.set_title('County-Level Mean Neighbor JSD', fontsize=TITLE_SIZE)
ax.axis('off')
plt.tight_layout()
out = FIGURES_EDA / 'county_mean_neighbor_jsd_map.svg'
plt.savefig(out, dpi=DPI, bbox_inches='tight', format='svg')
plt.close()
print(f'Saved: {out}')

Saved: c:\Users\sardo\OneDrive\Desktop\Classes\Wildfire-Property-Intelligence\figures\eda\in_depth_analysis\county_mean_neighbor_jsd_map.svg


## 2. County-Level Mean Surprisal

Uses `m01_neighbor_pool_county_lc_color_detail.csv`.

In [26]:
cp_detail = RESULTS / 'conditional_probability' / 'm01_neighbor_pool_county_lc_color_detail.csv'
df_detail = pd.read_csv(cp_detail)
df_detail['fips'] = df_detail['fips'].astype(str).str.zfill(5)
df_detail['surprisal'] = -np.log(np.maximum(df_detail['p_pool'], 1e-10))
county_mean_surp = df_detail.groupby('fips')['surprisal'].mean().reset_index(name='mean_surprisal')

gdf = ca.merge(county_mean_surp, left_on='county_fips', right_on='fips', how='left')
gdf['mean_surprisal'] = gdf['mean_surprisal'].fillna(gdf['mean_surprisal'].median())

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column='mean_surprisal', ax=ax, cmap='YlOrRd', legend=True, edgecolor=EDGE_COLOR, linewidth=BORDER_LW,
         legend_kwds={'label': 'Mean surprisal (nats)'})
ax.set_title('County-Level Mean Surprisal', fontsize=TITLE_SIZE)
ax.axis('off')
plt.tight_layout()
out = FIGURES_EDA / 'county_mean_surprisal_map.svg'
plt.savefig(out, dpi=DPI, bbox_inches='tight', format='svg')
plt.close()
print(f'Saved: {out}')

Saved: c:\Users\sardo\OneDrive\Desktop\Classes\Wildfire-Property-Intelligence\figures\eda\in_depth_analysis\county_mean_surprisal_map.svg


## 3. County Reliability Map (pct_reliable, pct_sparse)

Uses `bayesian_shrinkage_aggregated_counts.csv` and optionally dataset for pct_sparse.

In [27]:
bay_agg = RESULTS / 'bayesian_shrinkage' / 'bayesian_shrinkage_aggregated_counts.csv'
exp_by_county = RESULTS / '02_exposure_density_sparsity' / 'eda_exposure_by_county.csv'
exp_per_h3 = RESULTS / '02_exposure_density_sparsity' / 'eda_exposure_per_h3.csv'

df_bay = pd.read_csv(bay_agg)
cl_stats = df_bay.groupby(['fips', 'lc_type']).agg(total_structures=('count', 'sum')).reset_index()
pct_reliable = cl_stats.groupby('fips').apply(
    lambda g: 100 * (g['total_structures'] >= 50).sum() / len(g)
).reset_index(name='pct_reliable')
pct_reliable['fips'] = pct_reliable['fips'].astype(str).str.zfill(5)

county_stats = pd.read_csv(exp_by_county)
county_stats['county_fips'] = county_stats['county_fips'].astype(str).str.zfill(5)
county_stats = county_stats.merge(pct_reliable, left_on='county_fips', right_on='fips', how='left', suffixes=('', '_pr'))

# pct_sparse: needs h3->fips. Try dataset (usecols only)
h3_fips = None
for p in [PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz',
           PROJECT_ROOT / 'dataset' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv',
           PROJECT_ROOT / 'website' / 'backend' / 'data' / 'Capstone2025_nsi_lvl9_with_landcover_and_color.csv']:
    if p.exists():
        kwargs = {'compression': 'gzip', 'usecols': ['h3', 'fips']} if str(p).endswith('.gz') else {'usecols': ['h3', 'fips']}
        try:
            h3_fips = pd.read_csv(p, low_memory=False, **kwargs)[['h3', 'fips']].drop_duplicates()
            h3_fips['fips'] = h3_fips['fips'].astype(str).str.zfill(5)
            break
        except Exception:
            pass

if h3_fips is not None and len(h3_fips) > 0:
    df_exp = pd.read_csv(exp_per_h3)
    df_exp['sparse'] = df_exp['total_exposure'] < 10
    df_exp = df_exp.merge(h3_fips, on='h3', how='inner')
    pct_sparse = df_exp.groupby('fips')['sparse'].mean() * 100
    pct_sparse = pct_sparse.reset_index(name='pct_sparse')
    county_stats = county_stats.merge(pct_sparse, left_on='county_fips', right_on='fips', how='left', suffixes=('', '_ps'))
    county_stats['pct_sparse'] = county_stats['pct_sparse'].fillna(county_stats['pct_sparse'].median())
    n_axes = 2
else:
    n_axes = 1

gdf = ca.merge(county_stats, on='county_fips', how='left')
gdf['pct_reliable'] = gdf['pct_reliable'].fillna(gdf['pct_reliable'].median())

fig, axes = plt.subplots(1, n_axes, figsize=(7 * n_axes, 7))
axes = [axes] if n_axes == 1 else axes
gdf.plot(column='pct_reliable', ax=axes[0], cmap='YlOrRd', legend=True, edgecolor=EDGE_COLOR, linewidth=BORDER_LW)
axes[0].set_title('% County×Landcover Groups with ≥50 Structures', fontsize=TITLE_SIZE_DUAL)
axes[0].axis('off')
if n_axes > 1:
    gdf.plot(column='pct_sparse', ax=axes[1], cmap='YlOrRd', legend=True, edgecolor=EDGE_COLOR, linewidth=BORDER_LW)
    axes[1].set_title('% H3 Cells with Exposure < 10', fontsize=TITLE_SIZE_DUAL)
    axes[1].axis('off')
plt.tight_layout()
out = FIGURES_EDA / 'eda_county_reliability_map.svg'
plt.savefig(out, dpi=DPI, bbox_inches='tight', format='svg')
plt.close()
print(f'Saved: {out}')

C:\Users\sardo\AppData\Local\Temp\ipykernel_43884\3144214814.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pct_reliable = cl_stats.groupby('fips').apply(


Saved: c:\Users\sardo\OneDrive\Desktop\Classes\Wildfire-Property-Intelligence\figures\eda\in_depth_analysis\eda_county_reliability_map.svg


## 4. County-Level Mean JSD vs. Statewide Baseline

Uses `jsd_conditional_county_summary.csv` (or `jsd_conditional_divergence.csv`).

In [28]:
jsd_summary = RESULTS / 'grouplevel_divergence' / 'jsd_conditional_county_summary.csv'
jsd_div = RESULTS / 'grouplevel_divergence' / 'jsd_conditional_divergence.csv'

if jsd_summary.exists():
    county_jsd = pd.read_csv(jsd_summary)[['fips', 'avg_divergence']]
    county_jsd = county_jsd.rename(columns={'avg_divergence': 'mean_jsd'})
else:
    df = pd.read_csv(jsd_div)
    county_jsd = df.groupby('fips')['divergence'].mean().reset_index(name='mean_jsd')

county_jsd['fips'] = county_jsd['fips'].astype(str).str.zfill(5)

gdf = ca.merge(county_jsd, left_on='county_fips', right_on='fips', how='left')
gdf['mean_jsd'] = gdf['mean_jsd'].fillna(gdf['mean_jsd'].median())

cmap = LinearSegmentedColormap.from_list('jsd', ['#ffffcc', '#fd8d3c', '#e31a1c', '#67000d'])

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column='mean_jsd', ax=ax, cmap=cmap, edgecolor=EDGE_COLOR, linewidth=BORDER_LW,
         legend=True, legend_kwds={'label': 'Mean JSD (vs statewide baseline)'})
ax.set_title('County-Level Mean JSD vs. Statewide Baseline', fontsize=TITLE_SIZE)
ax.axis('off')
plt.tight_layout()
out = FIGURES_METHOD / 'county_jsd_baseline_map.svg'
plt.savefig(out, dpi=DPI, bbox_inches='tight', facecolor='white', format='svg')
plt.close()
print(f'Saved: {out}')

Saved: c:\Users\sardo\OneDrive\Desktop\Classes\Wildfire-Property-Intelligence\figures\method_comparison\county_jsd_baseline_map.svg


In [29]:
print('Done. All 4 maps saved as SVG.')

Done. All 4 maps saved as SVG.


## Outputs

The four existing SVG exports remain: `county_mean_neighbor_jsd_map.svg`, `county_mean_surprisal_map.svg`, `eda_county_reliability_map.svg`, and `county_jsd_baseline_map.svg`.
